In [1]:
"""
IRI Research Module - UPenn Ph.D. Project
Modular ODE solver for Ice Recrystallization Inhibition (IRI) kinetics.
"""
import numpy as np
from scipy.integrate import solve_ivp

def get_growth_velocity(R, c_bulk, params, mode='single'):
    """Calculates v_R based on current bulk concentration."""
    D = params['D']
    rho_ice = params['rho_ice']
    c_flat = params['c_flat']
    alpha = params['alpha']
    k1 = params.get('k1', params.get('k', 0))
    k2 = params.get('k2', 0)
    invL2 = params['invL2']

    if mode == 'single':
        return D / (R * rho_ice) * (c_bulk - c_flat - alpha / R - k1 * invL2) / 1000
    
    elif mode == 'double':
        # Condition for Two Critical Radii (Freezing Hysteresis)
        c1 = R > alpha / (c_bulk - c_flat - k1 * invL2)
        c2 = R < alpha / (c_bulk - c_flat + k2 * invL2)
        v_vals = [
            D / (R * rho_ice) * (c_bulk - c_flat - alpha / R - k1 * invL2) / 1000,
            D / (R * rho_ice) * (c_bulk - c_flat - alpha / R + k2 * invL2) / 1000
        ]
        return np.select([c1, c2], v_vals, default=0)

def ode_system(t, y, R, params, mode):
    """The system of ODEs: returns [df/dt, dc_bulk/dt]."""
    f = y[:-1]
    c_bulk = y[-1]
    rho_ice = params['rho_ice']
    
    v_R = get_growth_velocity(R, c_bulk, params, mode=mode)
    
    # Continuity Equation (Advection of PSD)
    df_dt = -np.gradient(f * v_R, R)
    
    # Conservation Equation (Mass balance of solution)
    # Using Simpson's rule for better integration accuracy than trapz
    from scipy.integrate import simpson
    dc_bulk_dt = -4 * np.pi * rho_ice * simpson(y=R * R * v_R * f, x=R)
    
    return np.concatenate([df_dt, [dc_bulk_dt]])

def run_simulation(f_init, c_bulk_init, R, t_span, params, mode='single'):
    """Professional BDF solver (equivalent to MATLAB ode15s)."""
    y0 = np.concatenate([f_init, [c_bulk_init]])
    
    # Solve with BDF method for stiff systems
    sol = solve_ivp(
        fun=ode_system,
        t_span=t_span,
        y0=y0,
        args=(R, params, mode),
        method='BDF', 
        rtol=1e-6, 
        atol=1e-9
    )
    return sol

In [2]:
# 1. Define the spatial grid (Radius)
R = np.linspace(0, 40, 401)[1:]  # From 0 to 40 micrometers, excluding 0 to avoid singularities

# 2. Set distribution parameters
n = 100             # Total number of crystals
V = 170**3          # Total volume of the system in cubic micrometers
mean_R = 10         # Mean radius of crystals (micrometers)
std_R = 2           # Standard deviation of the radius

# 3. Calculate the Gaussian distribution (f_init)
# The amplitude 'A' ensures the integral matches your total crystal count per volume
A = n / (std_R * V * np.sqrt(np.pi) * np.sqrt(2))
f_init = A * np.exp(-0.5 * ((R - mean_R) / std_R)**2)

In [4]:
import numpy as np

# 1. Required parameters (from your research constants)
delT = 5                                    # Temperature difference [K]
molL_to_microm = 6.02 * 10**8               # Unit conversion factor
mean_R = 10                                 # Mean radius from f_init [microm]

# 2. Calculate c_flat (Equilibrium concentration for a flat surface)
# Formula: c_flat = exp(-dH/RT * dT/Teq) * C_total
c_flat = np.exp(-6000/273 * delT / 8.314 / 268) * 55 * molL_to_microm

# 3. Calculate alpha (Gibbs-Thomson coefficient) and k (AFP term)
alpha = 2 * 30 / 1000 / 8.314 / 268 / 33 * 6.02 * 100 * c_flat
Pm = 0.005                                  # AFP parameter [microm]
k1 = alpha * Pm / 2                         # Engulfment parameter
invL2 = 1 / (0.1)**2                        # 1/L^2 term

# 4. Define c_bulk_init
# We set c_bulk to be slightly (0.1%) higher than the equilibrium 
# concentration for a particle with mean_R to initiate growth.
equilibrium_at_mean = c_flat + (alpha / mean_R) + (k1 * invL2)
c_bulk_init = equilibrium_at_mean * 1.001

print(f"Initial Bulk Concentration: {c_bulk_init:.4e}")

Initial Bulk Concentration: 3.1554e+10


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import iri_model



# 1. Setup Parameters (UPenn Research Constants)
params = {
    'D': 1120, 
    'rho_ice': 3.04e11, 
    'c_flat': 1.8e10, 
    'alpha': 5.5e9, 
    'k1': 38705, 
    'k2': 38705, 
    'invL2': 100, 
    'delt': 5
}

# 2. Run the ODE solver
# t_span defines [start_time, end_time] in ms
sol = iri_model.run_simulation(f_init, c_bulk_init, R, [0, 3600000], params, mode='double')

# 3. Access Results
# sol.t contains the adaptive time steps chosen by the solver
# sol.y contains the PSD and concentration data
final_f = sol.y[:-1, -1]
final_c_bulk = sol.y[-1, -1]